# Experiment: Qwen2.5-7B attribution patching on ObserverBench Copy-v2

**Question.** Does raw or scalar-calibrated attribution patching predict the frozen finite mean-replacement effects of eight Qwen2.5-7B induction-copy heads?

**Success criteria.** The CPU contract tests pass, an eight-prompt GPU smoke run completes, and the full run measures all 256 public training prompts before scoring both observers at budgets 16, 40, 64, and 128. Copy-v2's sealed Phase-10 artifacts are read but never changed. This is a retrospective published-method baseline, not a preregistered result.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


## Setup and reproducibility

Select an A100, H100, or H200 runtime with at least 40 GiB GPU memory. For a dated run, replace `main` with the commit containing this notebook and runner. Results go to Drive; the source checkout and frozen Phase-10 directory remain unchanged.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = os.environ.get('OBSERVERBENCH_REPO_URL', 'https://github.com/kwisatzh/observerbench.git')
REVISION = os.environ.get('OBSERVERBENCH_REVISION', 'main')
REPO_ROOT = Path('/content/ObserverBench')
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', REVISION], cwd=REPO_ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', REVISION], cwd=REPO_ROOT, check=True)

ARTIFACTS_ROOT = REPO_ROOT / 'results/revision/phase10/qwen_induction_copy_v2_complete/copy_v2'
RUNNER = REPO_ROOT / 'scripts/run_qwen_induction_attribution_patching_baseline.py'
CONSTRAINTS = REPO_ROOT / 'configs/revision/phase10/colab_constraints.txt'
OUTPUT_ROOT = Path('/content/drive/MyDrive/ObserverBenchArtifacts/published_baselines/qwen_attribution_patching_v1')
SMOKE_OUT = OUTPUT_ROOT / 'smoke'
FULL_OUT = OUTPUT_ROOT / 'full'
for path in (ARTIFACTS_ROOT / 'design/design_manifest.json', ARTIFACTS_ROOT / 'effects/effect_manifest.json', RUNNER, CONSTRAINTS):
    assert path.is_file(), f'Missing required input: {path}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
{'revision': REVISION, 'artifacts_root': str(ARTIFACTS_ROOT), 'output_root': str(OUTPUT_ROOT)}


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-c', str(CONSTRAINTS), '-e', f'{REPO_ROOT}[qwen]'],
    cwd=REPO_ROOT,
    check=True,
)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gpu_memory_gib >= 39, f'At least 40 GiB GPU memory is required; found {gpu_memory_gib:.1f}.'
{'gpu': gpu_name, 'gpu_memory_gib': round(gpu_memory_gib, 1), 'torch': torch.__version__}


## Plan

1. Run the CPU/unit path, including a tiny randomly initialized Qwen whose infinitesimal finite effect checks the AtP sign.
2. Run one eight-prompt smoke measurement. Stop and inspect its cards and summary.
3. Set `RUN_FULL = True` only after the smoke passes. The full run uses all 256 train prompts and cached held-out effects.
4. Build the four Qwen leaderboard panels from the completed summary.


In [ ]:
subprocess.run(
    [
        sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider',
        'tests/test_qwen_induction_attribution_patching.py',
        'tests/test_qwen_effect_leaderboards.py',
    ],
    cwd=REPO_ROOT,
    check=True,
)


## Minimal GPU smoke

The smoke run loads the pinned unquantized BF16 model but differentiates only eight train prompts. Its output is explicitly marked non-claim-eligible.


In [ ]:
smoke_command = [
    sys.executable, str(RUNNER),
    '--artifacts-root', str(ARTIFACTS_ROOT),
    '--outdir', str(SMOKE_OUT),
    '--device', 'cuda',
    '--batch-size', '2',
    '--max-prompts', '8',
]
subprocess.run(
    smoke_command, cwd=REPO_ROOT, check=True,
    env={**os.environ, 'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false'},
)
smoke_manifest = json.loads((SMOKE_OUT / 'run_manifest.json').read_text())
assert smoke_manifest['status'] == 'engineering_smoke'
assert smoke_manifest['claim_eligible'] is False
smoke_manifest['measurement']


## Full 256-prompt measurement

Review the smoke output before enabling this cell. No finite intervention is rerun: after the gradient map is written, scoring uses the frozen Copy-v2 tables.


In [ ]:
RUN_FULL = False
if RUN_FULL:
    full_command = [
        sys.executable, str(RUNNER),
        '--artifacts-root', str(ARTIFACTS_ROOT),
        '--outdir', str(FULL_OUT),
        '--device', 'cuda',
        '--batch-size', '4',
    ]
    subprocess.run(
        full_command, cwd=REPO_ROOT, check=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false'},
    )
    full_manifest = json.loads((FULL_OUT / 'run_manifest.json').read_text())
    assert full_manifest['status'] == 'complete'
    assert full_manifest['claim_eligible'] is True
    print((FULL_OUT / 'summary.csv').read_text())
else:
    print('Full run disabled. Inspect the smoke, then set RUN_FULL = True and rerun this cell.')


## Results and leaderboard panels

After the full run, this builds checked panels beside the result bundle. Copy them into `leaderboards/effect/` only when the completed run is ready for review.


In [ ]:
if RUN_FULL:
    leaderboard_out = FULL_OUT / 'leaderboards'
    subprocess.run(
        [
            sys.executable, 'scripts/build_qwen_effect_leaderboards.py',
            '--output-root', str(leaderboard_out),
            '--atp-path', str(FULL_OUT / 'summary.csv'),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
    print('Panels:', sorted(path.name for path in leaderboard_out.iterdir()))


## Next steps

- Check raw and calibrated MAE against the additive and quadratic rows at each budget.
- Report gradient and forward-only access separately; budget equality is not compute equality.
- Preserve the `post-outcome published-method baseline` label in cards, panels, and any paper text.
